# 11 · Streaming and Async

Two different performance problems, two different tools:

```
perceived latency  →  STREAMING  →  show tokens as they generate ("feels fast")
throughput         →  ASYNC      →  run independent calls concurrently ("is fast")
```

Everything in lessons 01–10 was built from Runnables — so everything already supports both. Same chains, one letter different.

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [ ]:
import os
import asyncio
import time
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.7)

---
## 1. Stream tokens from the model

`.stream()` — the Runnable interface from lesson 01, one letter away from `.invoke()`. First token in ~hundreds of ms instead of the full answer seconds later.

In [ ]:
for chunk in llm.stream("Explain in two sentences why chat UIs stream tokens."):
    print(chunk.content, end="", flush=True)

## 2. Chains stream end-to-end

LCEL components transform chunks *as they pass through* — `StrOutputParser` converts each chunk in-flight, so the whole pipe stays streaming.

In [ ]:
chain = (
    ChatPromptTemplate.from_template("Write a 3-line poem about {topic}.")
    | llm
    | StrOutputParser()
)

for chunk in chain.stream({"topic": "concurrency"}):
    print(chunk, end="", flush=True)

## 3. Streaming agents: three granularities

`stream_mode="updates"` → step-level progress ("calling tool X..."). `stream_mode="messages"` → token-level output for chat UIs. (`"values"` — full state per step — was lesson 06.)

In [ ]:
@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

agent = create_agent(llm, [word_count])
question = {"messages": [("human", "How many words are in 'streaming makes apps feel alive'?")]}

for update in agent.stream(question, stream_mode="updates"):
    for node, payload in update.items():
        msg = payload["messages"][-1]
        detail = msg.tool_calls[0]["name"] if getattr(msg, "tool_calls", None) else msg.content
        print(f"[{node}] {detail}")

In [ ]:
# Token-level: what you'd pipe to a chat UI.
for token, meta in agent.stream(question, stream_mode="messages"):
    if token.content and meta["langgraph_node"] == "model":
        print(token.content, end="", flush=True)

---
## 4. Async: the `a`-twins

Every sync method has an awaitable twin: `ainvoke`, `astream`, `abatch`. Notebooks support top-level `await`.

In [ ]:
reply = await llm.ainvoke("Say hello in exactly three words.")
print(reply.content)

## 5. The payoff: concurrency

Five independent calls. Sequential pays 5× latency; `asyncio.gather` pays ~1× (the slowest call) — the wait is network, not CPU.

In [ ]:
topics = ["Python", "Rust", "Go", "TypeScript", "SQL"]
prompts = [f"One-line fun fact about {t}." for t in topics]

t0 = time.perf_counter()
seq = [llm.invoke(p) for p in prompts]
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
conc = await asyncio.gather(*(llm.ainvoke(p) for p in prompts))
t_conc = time.perf_counter() - t0

print(f"sequential: {t_seq:.1f}s   concurrent: {t_conc:.1f}s   → {t_seq/t_conc:.1f}× faster\n")
for t, r in zip(topics, conc):        # gather preserves input order
    print(f"{t:>10}: {r.content}")

## 6. `abatch`: same idea, one runnable, many inputs

Plus `max_concurrency` to stay friendly with rate limits.

In [ ]:
results = await chain.abatch(
    [{"topic": t} for t in ["coffee", "debugging", "deadlines"]],
    config={"max_concurrency": 2},
)
print(results[0])

## 7. Async + streaming compose

In [ ]:
async for chunk in chain.astream({"topic": "the event loop"}):
    print(chunk, end="", flush=True)

---
## What to remember

| Concept | What it does |
|---|---|
| `.stream()` | Tokens as they generate — first token in ms, not the answer in seconds |
| LCEL streams through | Parsers transform chunks in-flight; the pipe stays streaming |
| `stream_mode="updates"` / `"messages"` | Agent progress events / token-level agent output |
| `ainvoke` / `abatch` / `astream` | Awaitable twins — same args, same results |
| `asyncio.gather(*coros)` | Independent calls concurrently; ~N× on network-bound work |
| `config={"max_concurrency": n}` | Throttle for rate limits |

**Streaming = feels fast. Async = is fast.** Production endpoints use both at once: `astream` the answer to the user while `gather`-ing side work (logging, memory writes, title generation).

**Next:** 12 — **Tracing and evaluation with LangSmith**: see inside everything you've built, and measure whether it's any good. *Series finale.*